In [1]:
# Part A — Creating the Date Spine
# Q1. Load the store dataset
import pandas as pd
import numpy as np

STORE_PATH = "store_transactions_sparse.csv"
ENERGY_PATH = "energy_usage_hourly.csv"
store = pd.read_csv(STORE_PATH,parse_dates=["date"])

print("Rows:", len(store))
print("Minimum date:", store["date"].min())
print("Maximum date:", store["date"].max())
print("="*50)
calendar_days = (store["date"].max() - store["date"].min()).days + 1
missing_days = calendar_days - len(store)
print("Calendar days:", calendar_days)
print("Difference:", missing_days)

Rows: 405
Minimum date: 2023-01-02 00:00:00
Maximum date: 2024-02-29 00:00:00
Calendar days: 424
Difference: 19


In [2]:
# Q2. Build a complete daily date spine
spine = pd.date_range(
    start=store["date"].min(),
    end=store["date"].max(),
    freq="D"
)

store_spined = (store.set_index("date").reindex(spine))
store_spined.index.name = "date"

print(store_spined.shape)
print("="*50)
print(store_spined.head())
print("="*30)
print(store_spined.isnull().sum())

(424, 2)
            transactions   revenue
date                              
2023-01-02         788.0  15680.68
2023-01-03         799.0  15184.39
2023-01-04         722.0  10945.52
2023-01-05         862.0  14243.50
2023-01-06         909.0  14807.19
transactions    19
revenue         19
dtype: int64


In [3]:
# Q3. Missing store days — 0 or interpolation?
store_spined["transactions"] = (store_spined["transactions"].fillna(0))
store_spined["revenue"] = (store_spined["revenue"].fillna(0))

# Store closed
#       ↓
# No transactions
#       ↓
# transactions = 0
# revenue = 0

In [4]:
# Q4. Add was_closed

store_spined = (store.set_index("date").reindex(spine))
store_spined.index.name = "date"
store_spined["was_closed"] = (store_spined["revenue"].isna())
store_spined["transactions"] = (store_spined["transactions"].fillna(0))
store_spined["revenue"] = (store_spined["revenue"].fillna(0))
print(store_spined["was_closed"].value_counts())

was_closed
False    405
True      19
Name: count, dtype: int64


In [5]:
# Q5. Energy hourly spine
energy = pd.read_csv(
    ENERGY_PATH,
    parse_dates=["timestamp"])

energy = energy.sort_values("timestamp")

hourly_spine = pd.date_range(
    start=energy["timestamp"].min(),
    end=energy["timestamp"].max(),
    freq="h"
)

energy_spined = (
    energy
    .set_index("timestamp")
    .reindex(hourly_spine)
)

energy_spined.index.name = "timestamp"

print("Expected hours:", len(hourly_spine))
print("Actual rows:", len(energy))
print("Missing hours:", energy_spined["kwh"].isna().sum())

Expected hours: 720
Actual rows: 709
Missing hours: 11


In [6]:
energy_spined["kwh"] = (
    energy_spined["kwh"]
    .interpolate(method="time")
)

In [7]:
# Part B — Lag and Lead Functions
# Q6. lag_1 and lag_7
store_spined["lag_1"] = (
    store_spined["revenue"].shift(1)
)

store_spined["lag_7"] = (
    store_spined["revenue"].shift(7)
)

In [8]:
# Q7. lead_1
store_spined["lead_1"] = (
    store_spined["revenue"].shift(-1)
)

In [9]:
# Q8. Day-over-day change
store_spined["dod_change"] = (
    store_spined["revenue"]
    - store_spined["lag_1"]
)

# percentage change
store_spined["dod_pct_change"] = (
    store_spined["revenue"].pct_change() * 100
)

In [10]:
# Q9. 365-day lag ወይስ 7-day lag?


In [11]:
# Q10. Record Day
store_spined["lag_14"] = (
    store_spined["revenue"].shift(14)
)

store_spined["record_day"] = (
    (store_spined["revenue"] > store_spined["lag_7"]) &
    (store_spined["revenue"] > store_spined["lag_14"])
)

In [12]:
# Part C — Rolling Windows
# Q11. 7-day trailing rolling mean

store_spined["rolling_7_mean"] = (
    store_spined["revenue"]
    .rolling(window=7)
    .mean()
)
print(store_spined["rolling_7_mean"].tail())

date
2024-02-25    20034.221429
2024-02-26    19904.468571
2024-02-27    19794.122857
2024-02-28    19737.992857
2024-02-29    20557.960000
Freq: D, Name: rolling_7_mean, dtype: float64


In [13]:
# Q12. Centered rolling mean
store_spined["rolling_7_centered"] = (
    store_spined["revenue"]
    .rolling(window=7, center=True)
    .mean()
)
print(store_spined["rolling_7_centered"])

date
2023-01-02             NaN
2023-01-03             NaN
2023-01-04             NaN
2023-01-05    14715.478571
2023-01-06    14231.422857
                  ...     
2024-02-25    19737.992857
2024-02-26    20557.960000
2024-02-27             NaN
2024-02-28             NaN
2024-02-29             NaN
Freq: D, Name: rolling_7_centered, Length: 424, dtype: float64


In [14]:
# Q13. Rolling standard deviation + anomaly flag
store_spined["rolling_7_std"] = (
    store_spined["revenue"]
    .rolling(window=7)
    .std()
)

store_spined["unusual"] = (
    abs(
        store_spined["revenue"]- store_spined["rolling_7_mean"])>2 * store_spined["rolling_7_std"]
)

print(store_spined["unusual"])

date
2023-01-02    False
2023-01-03    False
2023-01-04    False
2023-01-05    False
2023-01-06    False
              ...  
2024-02-25    False
2024-02-26    False
2024-02-27    False
2024-02-28    False
2024-02-29    False
Freq: D, Name: unusual, Length: 424, dtype: bool


In [15]:
# Q14. min_periods=1
# rolling(7)

# store_spined["rolling_7_mean_min1"] = (
#     store_spined["revenue"]
#     .rolling(window=7, min_periods=1)
#     .mean()
# )

In [16]:
# Q15. Expanding mean
store_spined["expanding_mean"] = (
    store_spined["revenue"]
    .expanding()
    .mean()
)

In [17]:
# Part D — Aggregating to Different Frequencies
# Q16. Daily → Weekly

weekly = (
    store_spined
    .resample("W")
    .agg(
        transactions=("transactions", "sum"),
        revenue=("revenue", "sum")
    )
)

print(weekly)

            transactions    revenue
date                               
2023-01-08        6117.0  103008.35
2023-01-15        6233.0  108047.18
2023-01-22        6413.0  117280.35
2023-01-29        6516.0  118372.03
2023-02-05        6340.0  112646.62
...                  ...        ...
2024-02-04        8086.0  159922.10
2024-02-11        8019.0  145180.08
2024-02-18        7969.0  139376.98
2024-02-25        7833.0  140239.55
2024-03-03        4238.0   73589.80

[61 rows x 2 columns]


In [18]:
# Q17. Average Transaction Value
monthly = (
    store_spined
    .resample("MS")
    .agg(
        revenue=("revenue", "sum"),
        transactions=("transactions", "sum")
    )
)

monthly["average_transaction_value"] = (
    monthly["revenue"]
    / monthly["transactions"]
)

In [19]:
# Q18. Energy Hourly → Daily → Weekly
# Daily Total
energy_daily = (
    energy_spined["kwh"]
    .resample("D")
    .sum()
)

# Weekly Total
energy_weekly = (
    energy_spined["kwh"]
    .resample("W")
    .sum()
)

# Daily → Hourly
hourly_from_daily = (
    energy_daily
    .resample("h")
    .asfreq()
)
hourly_from_daily.interpolate()

timestamp
2024-03-01 00:00:00    62.069000
2024-03-01 01:00:00    61.640917
2024-03-01 02:00:00    61.212833
2024-03-01 03:00:00    60.784750
2024-03-01 04:00:00    60.356667
                         ...    
2024-03-29 20:00:00    51.934833
2024-03-29 21:00:00    51.441625
2024-03-29 22:00:00    50.948417
2024-03-29 23:00:00    50.455208
2024-03-30 00:00:00    49.962000
Freq: h, Name: kwh, Length: 697, dtype: float64

In [20]:
# Q19. Average Revenue by Weekday
weekday_avg = (
    store_spined
    .groupby(store_spined.index.dayofweek)["revenue"]
    .mean()
)

weekday_avg.index = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_avg = weekday_avg.reset_index()
weekday_avg.columns = [
    "weekday",
    "average_revenue"
]

print(weekday_avg)

     weekday  average_revenue
0     Monday     15573.160328
1    Tuesday     16036.700164
2  Wednesday     16809.885410
3   Thursday     16720.768033
4     Friday     17900.058167
5   Saturday     20426.670333
6     Sunday     19808.142167


In [21]:
# Q20. Monthly Revenue — Two Methods
# Method A — Direct
monthly_direct = (
    store_spined["revenue"]
    .resample("MS")
    .sum()
)
# Method B — Weekly first
weekly_revenue = (
    store_spined["revenue"]
    .resample("W")
    .sum()
)

monthly_from_weekly = (
    weekly_revenue
    .resample("MS")
    .sum()
)

In [22]:
# Part E — Q21 Mini Integration Challenge
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

STORE_PATH = "store_transactions_sparse.csv"

store = pd.read_csv(
    STORE_PATH,
    parse_dates=["date"]
)

store = store.sort_values("date")


# ============================================================
# 2. CREATE COMPLETE DAILY DATE SPINE
# ============================================================

date_spine = pd.date_range(
    start=store["date"].min(),
    end=store["date"].max(),
    freq="D"
)

store_clean = (
    store
    .set_index("date")
    .reindex(date_spine)
)

store_clean.index.name = "date"


# ============================================================
# 3. IDENTIFY CLOSED / MISSING DAYS
# ============================================================

store_clean["was_closed"] = (
    store_clean["revenue"].isna()
)


# ============================================================
# 4. FILL CLOSED DAYS WITH ZERO
# ============================================================

store_clean["transactions"] = (
    store_clean["transactions"].fillna(0)
)

store_clean["revenue"] = (
    store_clean["revenue"].fillna(0)
)


# ============================================================
# 5. 7-DAY TRAILING ROLLING AVERAGE
# ============================================================

store_clean["rolling_7d_avg_revenue"] = (
    store_clean["revenue"]
    .rolling(window=7)
    .mean()
)


# ============================================================
# 6. RESAMPLE DAILY DATA TO WEEKLY
# ============================================================

weekly = (
    store_clean
    .resample("W")
    .agg(
        total_revenue=("revenue", "sum"),
        total_transactions=("transactions", "sum"),
        average_daily_revenue=("revenue", "mean")
    )
)


# ============================================================
# 7. 4-WEEK ROLLING AVERAGE OF WEEKLY REVENUE
# ============================================================

weekly["rolling_4w_avg_revenue"] = (
    weekly["total_revenue"]
    .rolling(window=4)
    .mean()
)


# ============================================================
# 8. WEEK-OVER-WEEK PERCENT CHANGE
# ============================================================

weekly["wow_pct_change"] = (
    weekly["total_revenue"]
    .pct_change()
    * 100
)


# ============================================================
# 9. RESET INDEX
# ============================================================

weekly = weekly.reset_index()


# ============================================================
# 10. DISPLAY FINAL TABLE
# ============================================================

print(weekly.to_string(index=False))

      date  total_revenue  total_transactions  average_daily_revenue  rolling_4w_avg_revenue  wow_pct_change
2023-01-08      103008.35              6117.0           14715.478571                     NaN             NaN
2023-01-15      108047.18              6233.0           15435.311429                     NaN        4.891671
2023-01-22      117280.35              6413.0           16754.335714                     NaN        8.545498
2023-01-29      118372.03              6516.0           16910.290000             111676.9775        0.930829
2023-02-05      112646.62              6340.0           16092.374286             114086.5450       -4.836793
2023-02-12      114535.85              6434.0           16362.264286             115708.7125        1.677130
2023-02-19      112104.06              6363.0           16014.865714             114414.6400       -2.123169
2023-02-26      119786.44              6712.0           17112.348571             114768.2425        6.852901
2023-03-05      119